# Comparisons with spatial and ideal-point models

Rebuild the current manuscript's model overview and benchmark tables from the included October results. Run all cells from a fresh kernel. This notebook checks the publication summaries against the original fit-level metric tables before exporting them; it does not refit a model.

**Inputs:** `benchmarks/published/`. **Outputs:** displayed dataframes and CSV/LaTeX tables in `benchmarks/tables/`.

To rerun estimation from the included fixed training matrices, validation choices, test choices and classification folds, follow `benchmarks/README.md`. The two-dimensional comparison and the one-dimensional EM-IRT timing analysis are separate experiments.


In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
BENCHMARKS = ROOT / "benchmarks"
SRC = BENCHMARKS / "published"
OUT = BENCHMARKS / "tables"
assert SRC.is_dir(), "Open from manuscript/ or manuscript/notebooks/."
OUT.mkdir(exist_ok=True)
COUNTRIES = ["chile", "france", "brazil"]
MODELS = ["B2", "B1", "B4", "B3", "W-NOMINATE"]
LABELS = {"B2": "DYNAMAP (B2)", "B1": "Constant-scale (B1)",
          "B4": "Flexible-head (B4)", "B3": "Matrix factorization (B3)",
          "W-NOMINATE": "W-NOMINATE (WN)", "EMIRT": "EM-IRT"}
# Shared citizen and original-proposal coordinates used through spatial differences.
# B4 has this representation but does not guarantee preference for the nearer item.
JOINT_DISTANCE_MAP = {"B2": "Yes", "B1": "Yes", "B4": "Yes", "B3": "No",
                      "W-NOMINATE": "No"}


def ordered(frame, models=MODELS):
    return frame.assign(
        country=pd.Categorical(frame.country, COUNTRIES, ordered=True),
        model=pd.Categorical(frame.model, models, ordered=True),
    ).sort_values(["country", "model"]).reset_index(drop=True)


def export(frame, name, fmt):
    display(frame)
    frame.to_csv(OUT / (name + ".csv"), index=False)
    alignment = "".join("l" if c in ["Country", "Model"] else
                        "c" if c == "Joint distance map" else "r" for c in frame.columns)
    frame.to_latex(OUT / (name + ".tex"), index=False, escape=True,
                   formatters=fmt, na_rep="--", column_format=alignment)



## Model definitions

A joint distance map here means citizens and original policy proposals share coordinates used through spatial differences. W-NOMINATE is spatial, but its outcome coordinates are specific to each unordered comparison. Matching data and dimensions does not match parameter counts or configuration searches.


In [2]:
overview = pd.read_csv(SRC / "benchmark_model_overview.csv", keep_default_na=False)
assert len(overview) == 7 and overview.Estimator.is_unique
assert overview["Joint map"].tolist() == ["Yes", "Yes", "Yes", "No", "No", "No", "No"]
display(overview)
overview.to_csv(OUT / "benchmark_model_overview.csv", index=False)
alignment = r'>{\raggedright\arraybackslash}p{0.06\textwidth}'
alignment += r'>{\raggedright\arraybackslash}p{0.20\textwidth}'
alignment += r'>{\raggedright\arraybackslash}p{0.36\textwidth}'
alignment += r'>{\raggedright\arraybackslash}p{0.10\textwidth}'
alignment += r'>{\raggedright\arraybackslash}p{0.19\textwidth}'
overview.to_latex(OUT / "benchmark_model_overview.tex", index=False, escape=True,
                  column_format=alignment)


,ID,Estimator,Choice representation,Joint map,Role in study
0,B2,DYNAMAP,Squared-distance difference multiplied by a le...,Yes,Main maps; 2D prediction; 1D resources; matche...
1,B1,Constant-scale DYNAMAP,Squared-distance difference multiplied by one ...,Yes,2D prediction; 1D resources.
2,B4,Flexible-head DYNAMAP,Shared exponential transformations of per-coor...,Yes,2D prediction; matched head ablation.
3,B3,Matrix factorization,Participant--proposal inner products with a bi...,No,2D prediction.
4,WN,W-NOMINATE,Participant ideal points and separate outcome ...,No,2D prediction and runtime.
5,--,Bayesian IRT,Probit model with participant ideal points and...,No,2D estimation diagnostics.
6,--,EM-IRT,Probit model with participant ideal points and...,No,1D resources and validation quality.


## Two-dimensional held-out prediction

Each country uses common training support and identical evaluated test choices. The neural entries average metrics across three final fits; they are not probability ensembles. Classification labels are used only after the preference models have been estimated. Accuracy gives half credit to exact probability ties.


In [3]:
summary = pd.read_csv(SRC / "summary/full_2d_comparison.csv")
summary = summary[summary.model.isin(MODELS)].copy()
raw = pd.read_csv(SRC / "three_country/predictive_comparison.csv")
raw = raw[raw.model.isin(MODELS)].copy()
external = pd.read_csv(SRC / "three_country/political_selfreport_comparison.csv")
external = external[external.model.isin(MODELS)].copy()
assert not summary.duplicated(["country", "model"]).any()
assert len(summary) == len(COUNTRIES) * len(MODELS)
checks = []
for r in summary.itertuples():
    subset = raw[(raw.country == r.country) & (raw.model == r.model)
                 & (raw.requested_size == "full") & (raw.aggregation == "mean_fit_metric")]
    assert len(subset) == 3, (r.country, r.model)
    for metric in ["accuracy", "log_loss", "brier"]:
        v = subset[subset.metric == metric].iloc[0]
        for weight in ["micro", "macro"]:
            assert np.isclose(getattr(r, weight + "_" + metric), v[weight], rtol=0, atol=1e-12)
    e = external[(external.country == r.country) & (external.model == r.model)
                 & (external.requested_size == "full")]
    assert len(e) == r.fits and e.people.nunique() == 1
    assert e.people.iloc[0] == r.labeled_people
    assert np.isclose(e.accuracy.mean(), r.selfreport_accuracy, atol=1e-12, rtol=0)
    checks.append({"country": r.country, "model": r.model, "predictive_people": int(v.people),
                   "test_rows": int(v.rows), "labeled_people": int(r.labeled_people), "fits": int(r.fits)})

d = ordered(summary)
table = pd.DataFrame({"Country": d.country.astype(str).str.title(),
                      "Model": d.model.astype(str).map(LABELS),
                      "Joint distance map": d.model.astype(str).map(JOINT_DISTANCE_MAP),
                      "Choice acc. (%)": d.micro_accuracy * 100,
                      "Person acc. (%)": d.macro_accuracy * 100,
                      "Person log loss": d.macro_log_loss,
                      "Label acc. (%)": d.selfreport_accuracy * 100})
export(table, "benchmark_2d_performance", {c: (lambda x: f"{x:.4f}") if c == "Person log loss"
                                          else (lambda x: f"{x:.2f}") for c in table.columns[3:]})


,Country,Model,Joint distance map,Choice acc. (%),Person acc. (%),Person log loss,Label acc. (%)
0,Chile,DYNAMAP (B2),Yes,70.802037,70.400585,0.568373,84.488189
1,Chile,Constant-scale (B1),Yes,70.630026,70.243614,0.571695,84.109799
2,Chile,Flexible-head (B4),Yes,69.781914,69.304004,0.581385,83.453631
3,Chile,Matrix factorization (B3),No,71.916485,71.629652,0.549149,85.643045
4,Chile,W-NOMINATE (WN),No,71.138008,70.729305,0.564884,84.724409
5,France,DYNAMAP (B2),Yes,83.574441,61.386878,0.742726,91.505969
6,France,Constant-scale (B1),Yes,83.957116,61.510972,0.746494,91.597796
7,France,Flexible-head (B4),Yes,81.223202,58.983326,0.738466,91.643710
8,France,Matrix factorization (B3),No,84.427131,61.849387,0.761045,91.873278
9,France,W-NOMINATE (WN),No,84.342501,62.674749,0.745501,91.735537


## Computation time

Times describe the measured workflows: three final DYNAMAP fits, its configuration search and one native W-NOMINATE fit. They do not isolate optimizer speed or establish a hardware-independent ratio.


In [4]:
runtime = pd.read_csv(SRC / "summary/full_2d_job_times.csv")
assert len(runtime) == 3 and runtime.country.is_unique
assert np.allclose(runtime.B2_three_full_fits_seconds + runtime.B2_country_tuning_seconds,
                   runtime.B2_tuning_plus_three_full_fits_seconds, atol=1e-9, rtol=0)
runtime = runtime.set_index("country").loc[COUNTRIES].reset_index()
rt = pd.DataFrame({"Country": runtime.country.str.title(),
                   "DYNAMAP: 3 fits": runtime.B2_three_full_fits_seconds / 60,
                   "Tuning": runtime.B2_country_tuning_seconds / 60,
                   "DYNAMAP total": runtime.B2_tuning_plus_three_full_fits_seconds / 60,
                   "W-NOMINATE: 1 fit": runtime.WN_one_fit_seconds / 60})
export(rt, "benchmark_2d_runtime", {c: lambda x: f"{x:.2f}" for c in rt.columns[1:]})


,Country,DYNAMAP: 3 fits,Tuning,DYNAMAP total,W-NOMINATE: 1 fit
0,Chile,23.192095,25.866395,49.058491,1241.998971
1,France,8.744758,24.179253,32.924011,31.055368
2,Brazil,1.678163,6.234461,7.912623,4.364437


## One-dimensional EM-IRT comparison

This separate analysis reports medians over three primary fits. It uses validation choices and two CPU threads. Its runtime figures must be read together with prediction quality; none of the 60 paired primary comparisons met the predefined absolute log-loss difference of 0.005.


In [5]:
em = pd.read_csv(SRC / "summary/full_1d_runtime.csv")
em_raw = pd.read_csv(SRC / "emirt_scaling/summary.csv")
assert len(em) == 9 and not em.duplicated(["country", "model"]).any()
for r in em.itertuples():
    rows = em_raw[(em_raw.country == r.country) & (em_raw.model == r.model)
                  & (em_raw["size"] == "full") & (em_raw.stage == "primary")]
    assert len(rows) == 3 and (rows.status == "COMPLETE").all()
    for source, target in [("wall_seconds", "wall_median_seconds"), ("max_rss_kib", "rss_median_kib"),
                           ("validation_accuracy", "validation_accuracy_median"),
                           ("validation_log_loss", "validation_loss_median")]:
        assert np.isclose(rows[source].median(), getattr(r, target), rtol=0, atol=1e-10)
    assert rows.n_users.nunique() == 1 and rows.n_users.iloc[0] == r.people
em = ordered(em, ["B2", "B1", "EMIRT"])
et = pd.DataFrame({"Country": em.country.astype(str).str.title(),
                   "Model": em.model.astype(str).map(LABELS), "People": em.people,
                   "Time (s)": em.wall_median_seconds, "RAM (MiB)": em.rss_median_kib / 1024,
                   "Val. acc. (%)": em.validation_accuracy_median * 100,
                   "Val. log loss": em.validation_loss_median})
export(et, "benchmark_1d_emirt", {"People": lambda x: f"{x:,}", "Time (s)": lambda x: f"{x:.1f}",
                                 "RAM (MiB)": lambda x: f"{x:.1f}", "Val. acc. (%)": lambda x: f"{x:.2f}",
                                 "Val. log loss": lambda x: f"{x:.4f}"})


,Country,Model,People,Time (s),RAM (MiB),Val. acc. (%),Val. log loss
0,Chile,DYNAMAP (B2),27163,512.69,771.171875,66.730198,0.599180
1,Chile,Constant-scale (B1),27163,96.04,738.195312,66.820161,0.606505
2,Chile,EM-IRT,27163,1094.75,6715.207031,70.362350,0.568487
3,France,DYNAMAP (B2),1188,83.18,693.183594,79.922703,0.439991
4,France,Constant-scale (B1),1188,102.98,684.683594,80.349908,0.438239
5,France,EM-IRT,1188,63.78,574.339844,81.528726,0.414448
6,Brazil,DYNAMAP (B2),535,44.15,520.929688,80.432688,0.414801
7,Brazil,Constant-scale (B1),535,49.22,501.746094,80.371376,0.426822
8,Brazil,EM-IRT,535,8.60,142.308594,83.261803,0.370013


## Paired uncertainty and Bayesian diagnostics

Paired intervals resample participants, conditional on the saved fits. They do not include uncertainty from refitting models. The four-chain Bayesian fits reached the iteration limit without satisfying all diagnostic families. They remain diagnostic results rather than receiving an imputed predictive score.


In [6]:
paired = pd.read_csv(SRC / "three_country/paired_contrasts.csv")
paired = paired[(paired.requested_size == "full") & (paired.model_a == "B2")
                & paired.model_a.isin(MODELS) & paired.model_b.isin(MODELS)]
display(paired)
paired.to_csv(OUT / "paired_B2_full_contrasts.csv", index=False)
quality = pd.read_csv(SRC / "emirt_scaling/paired_costs.csv")
assert len(quality) == 60 and not quality.quality_within_005.any()


,model_a,model_b,metric,estimand,a_minus_b,ci95_low,ci95_high,people,multiplicity,country,requested_size
132,B2,B3,log_loss,person_macro_mean_fit_metric,0.019223,0.017959,0.020411,27163,"pointwise, no familywise superiority decision",chile,full
133,B2,B3,brier,person_macro_mean_fit_metric,0.007298,0.006810,0.007760,27163,"pointwise, no familywise superiority decision",chile,full
134,B2,B3,accuracy,person_macro_mean_fit_metric,-0.012291,-0.013604,-0.010884,27163,"pointwise, no familywise superiority decision",chile,full
135,B2,B4,log_loss,person_macro_mean_fit_metric,-0.013012,-0.014435,-0.011539,27163,"pointwise, no familywise superiority decision",chile,full
136,B2,B4,brier,person_macro_mean_fit_metric,-0.005186,-0.005740,-0.004626,27163,"pointwise, no familywise superiority decision",chile,full
137,B2,B4,accuracy,person_macro_mean_fit_metric,0.010966,0.009607,0.012279,27163,"pointwise, no familywise superiority decision",chile,full
138,B2,W-NOMINATE,log_loss,person_macro_mean_fit_metric,0.003489,0.001883,0.005195,27163,"pointwise, no familywise superiority decision",chile,full
139,B2,W-NOMINATE,brier,person_macro_mean_fit_metric,0.001317,0.000700,0.001960,27163,"pointwise, no familywise superiority decision",chile,full
140,B2,W-NOMINATE,accuracy,person_macro_mean_fit_metric,-0.003287,-0.005006,-0.001671,27163,"pointwise, no familywise superiority decision",chile,full
222,B2,B3,log_loss,person_macro_mean_fit_metric,-0.018319,-0.027665,-0.009125,1141,"pointwise, no familywise superiority decision",france,full


In [7]:
diagnostics = pd.read_csv(SRC / "three_country/bayesian_size_diagnostics.csv")
sensitivity = pd.read_csv(SRC / "three_country/prior_anchor_sensitivity.csv")
assert len(diagnostics) == 10 and len(sensitivity) == 6
assert diagnostics.status.eq("NONCONVERGED_AT_ITERATION_CAP").all()
display(diagnostics)
diagnostics.to_csv(OUT / "bayesian_size_diagnostics.csv", index=False)
display(sensitivity)
sensitivity.to_csv(OUT / "prior_anchor_sensitivity.csv", index=False)


,country,requested_size,status,iterations,probability_convergence,coordinate_convergence,all_item_parameter_convergence,retained_people,retained_pairs
0,chile,300,NONCONVERGED_AT_ITERATION_CAP,30000,False,False,False,145,2178
1,chile,1000,NONCONVERGED_AT_ITERATION_CAP,30000,False,False,False,553,3774
2,chile,5000,NONCONVERGED_AT_ITERATION_CAP,30000,False,False,False,2873,4001
3,chile,15000,NONCONVERGED_AT_ITERATION_CAP,30000,False,False,False,8521,4005
4,chile,full,NONCONVERGED_AT_ITERATION_CAP,30000,True,False,False,27163,4005
5,france,300,NONCONVERGED_AT_ITERATION_CAP,30000,False,False,False,243,6706
6,france,1000,NONCONVERGED_AT_ITERATION_CAP,30000,False,False,False,829,6955
7,france,full,NONCONVERGED_AT_ITERATION_CAP,30000,False,False,False,1188,6931
8,brazil,300,NONCONVERGED_AT_ITERATION_CAP,30000,False,False,False,200,2106
9,brazil,full,NONCONVERGED_AT_ITERATION_CAP,30000,False,False,False,535,2180


,country,requested_size,variant,status
0,chile,5000,item_shrink4,NONCONVERGED_AT_ITERATION_CAP
1,chile,5000,alternate_anchors,NONCONVERGED_AT_ITERATION_CAP
2,france,1000,item_shrink4,NONCONVERGED_AT_ITERATION_CAP
3,france,1000,alternate_anchors,NONCONVERGED_AT_ITERATION_CAP
4,brazil,full,item_shrink4,NONCONVERGED_AT_ITERATION_CAP
5,brazil,full,alternate_anchors,NONCONVERGED_AT_ITERATION_CAP


## Cross-checks

The following checks compare the reported values with the included numerical source tables. Full native estimation can take days and is a separate operation.


In [8]:
validation = {"predictive_summary_matches_original_metrics": True,
              "external_classification_matches_saved_fit_average": True,
              "emirt_summary_matches_three_primary_fit_medians": True,
              "runtime_totals_match_components": True,
              "quality_equivalent_pairs_emirt": 0, "quality_comparisons_emirt": len(quality),
              "denominators": checks,
              "note": "No retraining. Point estimates are not ensembles. No missing Bayesian scores imputed."}
display(validation)
(BENCHMARKS / "validation.json").write_text(json.dumps(validation, indent=2) + "\n")

Out[8]: 2927


{'predictive_summary_matches_original_metrics': True,
 'external_classification_matches_saved_fit_average': True,
 'emirt_summary_matches_three_primary_fit_medians': True,
 'runtime_totals_match_components': True,
 'quality_equivalent_pairs_emirt': 0,
 'quality_comparisons_emirt': 60,
 'denominators': [{'country': 'brazil',
   'model': 'B1',
   'predictive_people': 443,
   'test_rows': 9704,
   'labeled_people': 356,
   'fits': 3},
  {'country': 'brazil',
   'model': 'B2',
   'predictive_people': 443,
   'test_rows': 9704,
   'labeled_people': 356,
   'fits': 3},
  {'country': 'brazil',
   'model': 'B3',
   'predictive_people': 443,
   'test_rows': 9704,
   'labeled_people': 356,
   'fits': 3},
  {'country': 'brazil',
   'model': 'B4',
   'predictive_people': 443,
   'test_rows': 9704,
   'labeled_people': 356,
   'fits': 3},
  {'country': 'brazil',
   'model': 'W-NOMINATE',
   'predictive_people': 443,
   'test_rows': 9704,
   'labeled_people': 356,
   'fits': 1},
  {'country': 'chile